# 🚀 Модуль 10. Урок 4. Batch-процессинг, расчет стоимости токенов и оптимизация затрат (FinOps LLM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_10_llm_api/lesson_04_batch_processing_and_costs.ipynb)

## 📌 Бизнес-контекст и методология
При обработке 1 000 000 отзывов клиентов вызов API в лоб может стоить тысячи долларов.
Стратегии оптимизации затрат (FinOps):
1. **Batch API**: скидка 50% при асинхронной пакетной обработке в течение 24 часов.
2. **Prompt Caching**: кэширование системного промпта снижает стоимость входных токенов на 75-90%.
3. **Моделирование размеров моделей**: 90% задач аналитики закрываются быстрыми моделями (mini / flash).

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Калькулятор стоимости обработки корпуса текстов
n_records = 500_000
tokens_per_input = 400
tokens_per_output = 50

# Цены за 1M токенов (ориентир 2026-2027)
price_input_per_m = 0.15   # $0.15 за 1M токенов вход
price_output_per_m = 0.60  # $0.60 за 1M токенов выход

total_input_tokens = n_records * tokens_per_input
total_output_tokens = n_records * tokens_per_output

cost_input = (total_input_tokens / 1_000_000) * price_input_per_m
cost_output = (total_output_tokens / 1_000_000) * price_output_per_m
total_cost_standard = cost_input + cost_output

# При использовании Batch API (-50%)
cost_batch_api = total_cost_standard * 0.50

print(f"Объем данных:                         {n_records:,} записей")
print(f"Всего входных токенов:               {total_input_tokens:,}")
print(f"Стоимость Real-time API:             ${total_cost_standard:.2f}")
print(f"Стоимость через Batch API (-50%):     ${cost_batch_api:.2f}")


Объем данных:                         500,000 записей
Всего входных токенов:               200,000,000
Стоимость Real-time API:             $45.00
Стоимость через Batch API (-50%):     $22.50


### 🎯 Практическое задание 1: Экономия при Prompt Caching
Если 300 из 400 входных токенов кэшируются со скидкой 80%, рассчитайте итоговую экономию.

In [3]:
### ВАШ КОД ЗДЕСЬ
cached_tokens = 300
uncached_tokens = 100

cached_input_cost = (n_records * cached_tokens / 1_000_000) * (price_input_per_m * 0.20)
uncached_input_cost = (n_records * uncached_tokens / 1_000_000) * price_input_per_m
total_caching_cost = cached_input_cost + uncached_input_cost + cost_output

print(f"Стоимость с Prompt Caching: ${total_caching_cost:.2f}")
assert total_caching_cost < total_cost_standard * 0.70, "Кэширование должно дать существенную экономию"
print("✅ Тест успешно пройден: FinOps оптимизация доказана расчетом!")


Стоимость с Prompt Caching: $27.00
✅ Тест успешно пройден: FinOps оптимизация доказана расчетом!


### 💼 Правило FinOps для аналитика
Всегда рассчитывайте бюджет до запуска пакетной обработки в цикле `for`.